# Agentic Data Quality Evaluator — Demo

**ISO/IEC 5259 series · Powered by Mistral AI**

This notebook walks through the full 9-agent pipeline:
1. Load & profile a dataset
2. Agent 1 — Domain analysis & characteristic prioritization
3. Agent 2 — Syntactic + semantic rule generation
4. Agents 3–8 — Per-characteristic evaluation (Accuracy, Completeness, Consistency, Diversity, Credibility, Currentness)
5. Agent 9 — Final report with AI Act Article 10 compliance notes

---
**Before running:** set your Mistral API key below, or export it as `MISTRAL_API_KEY`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.dirname(os.path.abspath('')))

# Set your API key here (or use an env var)
os.environ.setdefault('MISTRAL_API_KEY', 'YOUR_KEY_HERE')

import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Markdown, display
print('Setup complete.')

## 1 · Create a synthetic demo dataset

We generate a small network intrusion dataset that mimics CAN / IDS data so you can run the demo without downloading anything.
Swap `df` with your own dataset in the next section.

In [ ]:
from pathlib import Path
import tempfile

rng = np.random.default_rng(42)
n = 2000

can_ids  = rng.integers(0, 2048, n)
dlc      = rng.integers(0, 9, n)           # Data Length Code 0-8
payloads = rng.integers(0, 256, (n, 8))
timestamps = np.sort(rng.uniform(0, 60, n))

labels = np.zeros(n, dtype=int)
labels[rng.choice(n, size=150, replace=False)] = 1  # 7.5% attack

# Introduce some intentional quality issues
dlc[rng.choice(n, 40)] = -1      # out-of-range DLC (syntactic error)
labels[rng.choice(n, 20)] = -1   # unlabeled records

df = pd.DataFrame({
    'timestamp': timestamps,
    'can_id':    can_ids,
    'dlc':       dlc,
    **{f'd{i}': payloads[:, i] for i in range(8)},
    'label':     labels,
})

# Save to temp CSV
demo_csv = Path(tempfile.mktemp(suffix='.csv'))
df.to_csv(demo_csv, index=False)

print(f'Dataset shape: {df.shape}')
print(f'Label distribution: {df["label"].value_counts().to_dict()}')
df.head()

## 2 · Profile the dataset

In [ ]:
from dq_evaluator.tools.data_profiler import profile_dataset, profile_to_text

profile = profile_dataset(df)
profile_text = profile_to_text(profile)
print(profile_text)

## 3 · Configure the evaluation

In [ ]:
from mistralai.client import Mistral
from dq_evaluator.models.iso5259 import QualityCharacteristic

client = Mistral(api_key=os.environ['MISTRAL_API_KEY'])

DOMAIN_DESCRIPTION = """
CAN bus network traffic dataset for automotive intrusion detection.
Used to train and validate anomaly detection ML models that identify
flooding, fuzzy, and malfunction attacks in in-vehicle networks.
Must comply with ISO/SAE 21434 automotive cybersecurity requirements
and EU AI Act Article 10 data governance provisions.
"""

DATASET_NAME = "Demo CAN IDS Dataset"

DATASET_METADATA = {
    "source_name": "Synthetic demo (based on SAD dataset structure)",
    "creation_date": "2024",
    "collection_method": "Synthetic generation from real vehicle CAN traces",
    "license": "CC BY 4.0",
}

# Optional: override agent-determined priorities (1=highest, 3=lowest)
PRIORITY_OVERRIDES = {
    QualityCharacteristic.ACCURACY:     1,
    QualityCharacteristic.COMPLETENESS: 1,
    QualityCharacteristic.DIVERSITY:    2,
    QualityCharacteristic.CONSISTENCY:  2,
    QualityCharacteristic.CREDIBILITY:  3,
    QualityCharacteristic.CURRENTNESS:  3,
}

print('Configuration ready.')

## 4 · Agent 1: Domain Analysis & Prioritization

In [ ]:
from dq_evaluator.agents.domain_analyst import analyze_domain

context, subject, auto_priorities = analyze_domain(
    client, DOMAIN_DESCRIPTION, profile_text
)

print(f'Domain  : {context.domain}')
print(f'Purpose : {context.purpose}')
print(f'Task    : {context.task_type}')
print(f'Standards: {context.standards}')
print(f'\nData subjects: {subject.entities}')
print(f'\nAuto-detected priorities:')
for c, p in sorted(auto_priorities.items(), key=lambda x: x[1]):
    print(f'  {c.value}: {p}')

## 5 · Agent 2: Rule Generation

In [ ]:
from dq_evaluator.agents.rule_generator import generate_rules

syntactic_rules, semantic_rules = generate_rules(client, context, profile_text)

print(f'Generated {len(syntactic_rules)} syntactic rules:')
for r in syntactic_rules:
    print(f'  [{r.rule_type}] {r.field}: {r.description}')
    if r.standard_reference:
        print(f'          ref: {r.standard_reference}')

print(f'\nGenerated {len(semantic_rules)} semantic rules:')
for r in semantic_rules:
    print(f'  [{r.rule_type}] {r.name}: {r.description}')

## 6 · Agents 3–8: Evaluate All Characteristics

In [ ]:
from dq_evaluator.agents.evaluators.accuracy    import evaluate_accuracy
from dq_evaluator.agents.evaluators.completeness import evaluate_completeness
from dq_evaluator.agents.evaluators.consistency import evaluate_consistency
from dq_evaluator.agents.evaluators.diversity   import evaluate_diversity
from dq_evaluator.agents.evaluators.credibility import evaluate_credibility
from dq_evaluator.agents.evaluators.currentness import evaluate_currentness
from dq_evaluator.main import build_context_text

priorities = {**auto_priorities, **PRIORITY_OVERRIDES}
for c in QualityCharacteristic:
    priorities.setdefault(c, 2)

context_text = build_context_text(context, subject)
results = []

print('Evaluating Accuracy ...')
results.append(evaluate_accuracy(
    client, df, profile_text, syntactic_rules, semantic_rules,
    context_text, priorities[QualityCharacteristic.ACCURACY]
))

print('Evaluating Completeness ...')
results.append(evaluate_completeness(
    client, df, profile, profile_text, context_text,
    priorities[QualityCharacteristic.COMPLETENESS]
))

print('Evaluating Consistency ...')
results.append(evaluate_consistency(
    client, df, profile_text, semantic_rules, context_text,
    priorities[QualityCharacteristic.CONSISTENCY]
))

print('Evaluating Diversity ...')
results.append(evaluate_diversity(
    client, df, profile, profile_text, context_text,
    priorities[QualityCharacteristic.DIVERSITY]
))

print('Evaluating Credibility ...')
results.append(evaluate_credibility(
    client, profile_text, context_text, DATASET_METADATA,
    priorities[QualityCharacteristic.CREDIBILITY]
))

print('Evaluating Currentness ...')
results.append(evaluate_currentness(
    client, profile_text, context_text, DATASET_METADATA,
    priorities[QualityCharacteristic.CURRENTNESS]
))

print('All evaluations complete.')

## 7 · Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: radar / bar chart of characteristic scores ──
ax = axes[0]
names  = [r.characteristic.value.title() for r in results]
scores = [r.overall_score if r.overall_score is not None else 0 for r in results]
prios  = [r.priority for r in results]
colors = ['#2196F3' if p == 1 else '#FF9800' if p == 2 else '#9E9E9E' for p in prios]

bars = ax.barh(names, scores, color=colors, edgecolor='white', height=0.6)
ax.set_xlim(0, 105)
ax.set_xlabel('Score (%)')
ax.set_title('Quality Characteristic Scores', fontweight='bold')
for bar, score in zip(bars, scores):
    ax.text(score + 1, bar.get_y() + bar.get_height()/2,
            f'{score:.0f}%', va='center', fontsize=9)

legend_patches = [
    mpatches.Patch(color='#2196F3', label='Priority 1 (highest)'),
    mpatches.Patch(color='#FF9800', label='Priority 2'),
    mpatches.Patch(color='#9E9E9E', label='Priority 3'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=8)

# ── Right: heatmap of all dimension scores ──
ax2 = axes[1]
dim_data, dim_labels, char_labels = [], [], []
for r in results:
    row = []
    for d in r.dimensions:
        s = d.score if isinstance(d.score, float) else None
        row.append(s if s is not None else float('nan'))
    dim_data.append(row)
    char_labels.append(r.characteristic.value[:4].upper())

import numpy as np
max_dims = max(len(r.dimensions) for r in results)
matrix = np.full((len(results), max_dims), np.nan)
ytick_labels = []
for i, r in enumerate(results):
    for j, d in enumerate(r.dimensions):
        s = d.score if isinstance(d.score, (int, float)) else None
        if s is not None:
            matrix[i, j] = float(s)
    ytick_labels.append(r.characteristic.value.title())

im = ax2.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=100)
ax2.set_yticks(range(len(results)))
ax2.set_yticklabels(ytick_labels)
ax2.set_xticks(range(max_dims))
ax2.set_xticklabels([f'D{i+1}' for i in range(max_dims)], fontsize=8)
ax2.set_title('Dimension Heatmap (green=high, red=low)', fontweight='bold')
plt.colorbar(im, ax=ax2, label='Score (%)')

for i in range(len(results)):
    for j in range(max_dims):
        v = matrix[i, j]
        if not np.isnan(v):
            ax2.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=7, color='black')

plt.tight_layout()
plt.savefig('dq_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to dq_results.png')

## 8 · Agent 9: Generate Final Report

In [ ]:
from dq_evaluator.agents.report_generator import generate_report, format_report_text

report = generate_report(client, DATASET_NAME, context, subject, results)
report_text = format_report_text(report)

# Save to file
Path('dq_report.md').write_text(report_text)
print('Report saved to dq_report.md')

display(Markdown(report_text))

## 9 · Inspect Generated Rules

In [ ]:
import json
from dq_evaluator.tools.rule_checker import check_syntactic_rules

syntactic_check = check_syntactic_rules(df, syntactic_rules)

rows = []
for key, res in syntactic_check.items():
    rows.append({
        'Rule': key,
        'Type': res.get('rule_type', ''),
        'Violations': res.get('violations', 'N/A'),
        'Pass Rate': f"{res.get('pass_rate', 0)*100:.1f}%" if res.get('pass_rate') is not None else 'N/A',
        'Description': res.get('description', ''),
    })

pd.DataFrame(rows).style.background_gradient(
    subset=['Pass Rate'],
    cmap='RdYlGn'
)

## 10 · Run on Your Own Dataset

Use the one-call convenience wrapper from `main.py`:

In [ ]:
# from dq_evaluator.main import run_evaluation
#
# report_text = run_evaluation(
#     dataset_path='/path/to/your/data.csv',
#     domain_description='Describe your domain and use case here',
#     dataset_name='My Dataset',
#     dataset_metadata={
#         'source_name': 'My Organisation',
#         'creation_date': '2024',
#         'license': 'CC BY 4.0',
#     },
#     output_path='my_report.md',
# )
# display(Markdown(report_text))